# Snowflake Model Registry Demo

Upload a pre-trained XGBoost model artifact (pickle) to Snowflake's Model Registry.

## 1. Connect to Snowflake

In [ ]:
from snowflake.snowpark import Session

CONNECTION_PARAMS = {
    "account": "<your_account>",
    "user": "<your_user>",
    "password": "<your_password>",
    "role": "ACCOUNTADMIN",
    "warehouse": "COMPUTE_WH",
}

session = Session.builder.configs(CONNECTION_PARAMS).create()
print(f"Connected as {session.get_current_role()}")

## 2. Create Database and Registry Schema

In [ ]:
session.sql("CREATE DATABASE IF NOT EXISTS ML_REGISTRY_DEMO").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS ML_REGISTRY_DEMO.REGISTRY").collect()
session.sql("USE DATABASE ML_REGISTRY_DEMO").collect()
session.sql("USE SCHEMA REGISTRY").collect()
print("Registry schema ready: ML_REGISTRY_DEMO.REGISTRY")

## 3. Load the Model Artifact

In [ ]:
import pickle
from pathlib import Path

MODEL_PATH = Path("model_artifact/xgboost_model.pkl")

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

print(f"Loaded model from {MODEL_PATH}")
print(f"Model type: {type(model).__name__}")

## 4. Prepare Sample Input Data

The registry needs sample input to infer the model's input signature.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, _ = load_breast_cancer(return_X_y=True, as_frame=True)
_, X_sample, _, _ = train_test_split(X, X.iloc[:, 0], test_size=0.1, random_state=42)

print(f"Sample data shape: {X_sample.shape}")
X_sample.head()

## 5. Register the Model

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="ML_REGISTRY_DEMO", schema_name="REGISTRY")

mv = reg.log_model(
    model,
    model_name="xgboost_breast_cancer",
    version_name="v1",
    conda_dependencies=["xgboost", "scikit-learn"],
    sample_input_data=X_sample,
    comment="XGBoost classifier trained on sklearn breast cancer dataset",
)

print("Model registered: xgboost_breast_cancer/v1")

## 6. Run Inference via the Registry

In [ ]:
results = mv.run(X_sample.head(5))
results

## 7. Explore the Registry

In [ ]:
print("Models in registry:")
reg.show_models()

In [ ]:
print("Available methods:")
mv.show_functions()

In [ ]:
session.close()
print("Done.")